## Comprehensive One-Step CNN Implementation Plan for Flood Risk Modeling

**Project Goal:** Develop a CNN model to predict flood risk on a grid across Rio de Janeiro every 15 minutes.

**Core Idea:**
Use historical rainfall patterns (as sequences of 2D grids) and potentially static geographical features to predict a 2D grid of flood likelihood for the next 15-minute interval. Flood occurrences will be used to generate target labels, acknowledging their imprecision and under-reporting.

---

### **Notebook Preamble & Setup (Markdown Cell)**
# Flood Risk Prediction with Convolutional Neural Networks

This notebook implements a Convolutional Neural Network (CNN) to predict flood risk on a grid across Rio de Janeiro.

**Methodology:**
1.  **Data Loading & Preprocessing:** Load rainfall data, flood occurrences, and city boundary.
2.  **Grid Definition:** Create a regular grid over the city.
3.  **Feature Engineering:**
    *   Interpolate rainfall onto the grid for each 15-minute interval.
    *   Create sequences of rainfall maps as input features.
    *   (Future Work: Incorporate static features like elevation, land use).
4.  **Target Variable Generation:**
    *   Rasterize flood occurrences (points) onto the grid, considering spatial and temporal buffers to account for imprecision.
5.  **Data Preparation for CNN:** Create input tensors (sequences of rainfall grids) and target tensors (flood/no-flood grids).
6.  **Model Architecture:** Define a ConvLSTM or 3D CNN architecture suitable for spatio-temporal data.
7.  **Model Training:** Train the model using a chronological data split.
8.  **Model Evaluation:** Assess performance using appropriate metrics for imbalanced data and spatial predictions.


---

### **1. Imports and Environment Setup (Python Cell)**

In [1]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import timedelta

# GIS and Raster processing
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize

# Interpolation
from scipy.interpolate import griddata # For IDW or other grid interpolation

# Machine Learning - TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, ConvLSTM2D, BatchNormalization, Activation, Dense, Reshape, Flatten, TimeDistributed, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split # For initial non-temporal split if needed, but primarily chronological
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

# Silence TensorFlow warnings (optional)
# import logging
# logging.getLogger('tensorflow').setLevel(logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # 0 = all messages, 1 = INFO, 2 = WARNING, 3 = ERROR

2025-06-18 10:13:16.672500: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-18 10:13:16.811170: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-18 10:13:16.909488: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750252397.031189  271173 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750252397.066240  271173 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750252397.323182  271173 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

---

### **2. Configuration and Constants (Python Cell)**

In [2]:
# --- CRS Definitions ---
CRS_GEOGRAPHIC = "EPSG:4326"  # WGS84 for input lat/lon
CRS_PROJECTED = "EPSG:31983"  # SIRGAS 2000 / UTM zone 23S

# --- Data Paths (from previous script, ensure they are accessible) ---
INPUT_DATA_DIRECTORY = '../../../../data/meteorologia/clean' # Adjust if needed
STATIONS_CSV_PATH = f'{INPUT_DATA_DIRECTORY}/clima_pluviometro/estacoes_alertario.csv'
CITY_BOUNDARY_FILE_PATH = f'{INPUT_DATA_DIRECTORY}/limites_geograficos_rj/limite_municipio_rio_de_janeiro.gpkg'
ALERTARIO_CSV_PATH = f'{INPUT_DATA_DIRECTORY}/clima_pluviometro/taxa_precipitacao_alertario_manual.csv'
OCORRENCIAS_CSV_PATH = f'{INPUT_DATA_DIRECTORY}/adm_cor_comando/ocorrencias.csv'
POPS_CSV_PATH = '../../data/raw/adm_cor_comando/pops.csv' # Adjust if needed

# --- Grid Parameters ---
GRID_RESOLUTION_METERS = 500 # Spatial resolution of each grid cell (e.g., 500m x 500m)
# GRID_HEIGHT and GRID_WIDTH will be determined dynamically from city bounds and resolution

# --- Temporal Parameters for CNN ---
TIME_RESOLUTION_MINUTES = 15
INPUT_SEQUENCE_LENGTH = 4 # Number of past 15-min rainfall maps to use as input (e.g., 4 = 1 hour)
# OUTPUT_SEQUENCE_LENGTH = 1 # Predicting the next 15-min interval

# --- Flood Labeling Parameters ---
FLOOD_SPATIAL_RADIUS_METERS = 750 # Radius around a flood point to mark cells as flooded
FLOOD_TEMPORAL_BUFFER_MINUTES_BEFORE = 30 # Time buffer before reported start
FLOOD_TEMPORAL_BUFFER_MINUTES_AFTER = 90  # Time buffer after reported end (to account for duration and imprecision)

# --- Model & Training Parameters ---
MODEL_OUTPUT_PATH = './cnn_flood_model.keras'
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
VALIDATION_SPLIT_RATIO = 0.2 # For chronological split, this might be a fixed date
TEST_SPLIT_RATIO = 0.2       # For chronological split

# --- Feature Scaling ---
# For rainfall, MinMax scaling is often appropriate, or standardization if distribution is Gaussian
RAINFALL_SCALER = MinMaxScaler() # or StandardScaler()

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

---

### **3. Load Data (Python Cell)**

### 3.1 Load Base Dataframes
Load `alertario`, `stations`, `ocorrencias`, `pops`, and `city_boundary_gdf` as done in the previous script.
Ensure data types are correct (especially timestamps) and initial filtering (e.g., flood types) is applied.
Project `stations_gdf` and `city_boundary_gdf` to `CRS_PROJECTED`.

In [3]:
# --- Load Rainfall Data (alertario) ---
alertario_df = pd.read_csv(ALERTARIO_CSV_PATH)
alertario_df['timestamp'] = pd.to_datetime(alertario_df['timestamp'])
alertario_df['acumulado_chuva_15_min'] = pd.to_numeric(alertario_df['acumulado_chuva_15_min'], errors='coerce').fillna(0)

# --- Load Station Locations ---
stations_df = pd.read_csv(STATIONS_CSV_PATH)
stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df.longitude, stations_df.latitude),
    crs=CRS_GEOGRAPHIC
).dropna(subset=['latitude', 'longitude']).drop_duplicates(subset=['id_estacao'])
stations_gdf = stations_gdf.to_crs(CRS_PROJECTED)

# Merge station coordinates into alertario_df
alertario_df = alertario_df.merge(stations_df[['id_estacao', 'latitude', 'longitude']], on='id_estacao', how='left')
alertario_df = alertario_df.dropna(subset=['latitude', 'longitude']) # Ensure all rainfall records have coords

# --- Load City Boundary ---
city_boundary_gdf = gpd.read_file(CITY_BOUNDARY_FILE_PATH).to_crs(CRS_PROJECTED)
study_area_polygon = city_boundary_gdf.geometry.unary_union
study_area_bounds = study_area_polygon.bounds # (minx, miny, maxx, maxy)

# --- Load Flood Occurrences ---
ocorrencias_df = pd.read_csv(OCORRENCIAS_CSV_PATH)
pops_df = pd.read_csv(POPS_CSV_PATH)
ocorrencias_df['data_inicio'] = pd.to_datetime(ocorrencias_df['data_inicio'])
ocorrencias_df['data_fim'] = pd.to_datetime(ocorrencias_df['data_fim'], errors='coerce')
# Handle missing end times: assume a duration if necessary, e.g., 2 hours
default_duration = timedelta(hours=2)
ocorrencias_df['data_fim'] = ocorrencias_df['data_fim'].fillna(ocorrencias_df['data_inicio'] + default_duration)

ocorrencias_df['tipo'] = ocorrencias_df['id_pop'].map(pops_df.set_index('id')['titulo'])
flood_types = ["Bolsão d'água em via", 'Alagamentos e enchentes'] # As per previous script
ocorrencias_df = ocorrencias_df[ocorrencias_df['tipo'].isin(flood_types)]

ocorrencias_gdf = gpd.GeoDataFrame(
    ocorrencias_df,
    geometry=gpd.points_from_xy(ocorrencias_df.longitude, ocorrencias_df.latitude),
    crs=CRS_GEOGRAPHIC
).to_crs(CRS_PROJECTED)
ocorrencias_gdf = ocorrencias_gdf.dropna(subset=['geometry']) # Remove events with no valid geometry

# Filter alertario by time range of occurrences (optional, but good for focus)
min_ocorrencia_time = ocorrencias_df['data_inicio'].min() - timedelta(days=INPUT_SEQUENCE_LENGTH * TIME_RESOLUTION_MINUTES / (24*60) + 1) # Ensure we have prior rainfall
max_ocorrencia_time = ocorrencias_df['data_fim'].max() + timedelta(days=1)

alertario_df = alertario_df[(alertario_df['timestamp'] >= min_ocorrencia_time.to_datetime64()) & (alertario_df['timestamp'] <= max_ocorrencia_time.to_datetime64())]

print(f"Rainfall data from {alertario_df['timestamp'].min()} to {alertario_df['timestamp'].max()}")
print(f"Flood occurrences from {ocorrencias_gdf['data_inicio'].min()} to {ocorrencias_gdf['data_fim'].max()}")
print(f"Number of rainfall records: {len(alertario_df)}")
print(f"Number of flood occurrence records: {len(ocorrencias_gdf)}")
print(f"Number of stations with rainfall data: {alertario_df['id_estacao'].nunique()}")

/tmp/ipykernel_271173/1771107790.py:21: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  study_area_polygon = city_boundary_gdf.geometry.unary_union


Rainfall data from 2015-04-09 15:00:00 to 2024-03-27 10:45:00
Flood occurrences from 2015-04-10 15:59:39+00:00 to 2024-03-26 10:48:33+00:00
Number of rainfall records: 10406927
Number of flood occurrence records: 4389
Number of stations with rainfall data: 33


In [4]:
alertario_df.to_csv(f'{INPUT_DATA_DIRECTORY}/clima_pluviometro/taxa_precipitacao_alertario_manual_corte_ocorrencias.csv', index=False)

---

### **4. Grid Definition & Feature Rasterization (Python Cell)**
### 4.1 Define City Grid
Create a regular grid covering the `study_area_polygon`.

In [4]:
minx, miny, maxx, maxy = study_area_bounds
GRID_WIDTH_CELLS = int(np.ceil((maxx - minx) / GRID_RESOLUTION_METERS))
GRID_HEIGHT_CELLS = int(np.ceil((maxy - miny) / GRID_RESOLUTION_METERS))

print(f"Grid dimensions: {GRID_HEIGHT_CELLS} (height) x {GRID_WIDTH_CELLS} (width) cells")

# Affine transform for rasterization
# Origin is top-left, so maxy for y-coordinate start
transform = from_origin(minx, maxy, GRID_RESOLUTION_METERS, GRID_RESOLUTION_METERS)

# Create a mask for the study area on the grid
city_mask_raster = rasterize(
    [(study_area_polygon, 1)], # Burn value 1 for city area
    out_shape=(GRID_HEIGHT_CELLS, GRID_WIDTH_CELLS),
    transform=transform,
    fill=0, # Fill value for outside city
    dtype='uint8'
)
# Invert mask: True for valid city cells, False otherwise
city_mask_bool = city_mask_raster.astype(bool)

print(f"Number of active grid cells within city: {np.sum(city_mask_bool)}")

# Get coordinates of grid cell centers (for IDW)
x_coords = np.arange(minx + GRID_RESOLUTION_METERS / 2, maxx, GRID_RESOLUTION_METERS)
y_coords = np.arange(maxy - GRID_RESOLUTION_METERS / 2, miny, -GRID_RESOLUTION_METERS) # y decreases from top
grid_x, grid_y = np.meshgrid(x_coords, y_coords)

# Filter grid cell centers to be within the study area polygon for efficiency
grid_points = [Point(x, y) for x, y in zip(grid_x.ravel(), grid_y.ravel())]
grid_cell_centers_gdf = gpd.GeoDataFrame(geometry=grid_points, crs=CRS_PROJECTED)
# Sjoin is slow for many points. Use the raster mask instead.
valid_grid_indices_flat = np.where(city_mask_bool.ravel())[0]
grid_x_flat_valid = grid_x.ravel()[valid_grid_indices_flat]
grid_y_flat_valid = grid_y.ravel()[valid_grid_indices_flat]

Grid dimensions: 75 (height) x 144 (width) cells
Number of active grid cells within city: 4809


### 4.2 Interpolate Rainfall onto Grid (IDW)
For each 15-minute timestamp with rainfall data, interpolate station readings (`acumulado_chuva_15_min`) onto the grid.

In [ ]:
def interpolate_rainfall_to_grid_idw(timestamp_data, target_grid_x_flat, target_grid_y_flat, city_mask_raster_bool_flat, power=2):
    """Interpolates rainfall for a single timestamp using IDW."""
    station_points = np.array(timestamp_data[['longitude_proj', 'latitude_proj']].values)
    rainfall_values = timestamp_data['acumulado_chuva_15_min'].values

    # If no stations have data, or only one, handle appropriately
    if len(station_points) == 0:
        return np.zeros_like(target_grid_x_flat, dtype=float)
    if len(station_points) == 1:
        # Assign uniform value or handle as per choice. For simplicity, assign to all valid cells.
        interpolated_values_flat = np.full_like(target_grid_x_flat, rainfall_values[0], dtype=float)
        return interpolated_values_flat

    # griddata expects target points as (N,2) array
    target_points = np.vstack((target_grid_x_flat, target_grid_y_flat)).T
    
    try:
        interpolated_values_flat = griddata(
            station_points,
            rainfall_values,
            target_points,
            method='linear', # 'linear', 'nearest', 'cubic' (cubic can be slow and produce extremes)
            fill_value=0 # Fill value for points outside convex hull of stations
        )
        interpolated_values_flat[interpolated_values_flat < 0] = 0 # rainfall cannot be negative
    except Exception as e: # Catch potential qhull errors with insufficient points for cubic
        # Fallback to linear or nearest if cubic fails
        # print(f"Cubic interpolation failed for timestamp, falling back to linear: {e}")
        try:
            interpolated_values_flat = griddata(
                station_points,
                rainfall_values,
                target_points,
                method='linear',
                fill_value=0
            )
            interpolated_values_flat[interpolated_values_flat < 0] = 0
        except:
            # print(f"Linear interpolation also failed, falling back to nearest:")
            interpolated_values_flat = griddata(
                station_points,
                rainfall_values,
                target_points,
                method='nearest',
                fill_value=0
            )


    # Ensure NaNs that might arise (e.g. if all stations are far) are 0
    interpolated_values_flat = np.nan_to_num(interpolated_values_flat)
    return interpolated_values_flat


'''
# Prepare station coordinates in projected CRS for interpolation
temp_stations_gdf = gpd.GeoDataFrame(
    alertario_df[['longitude', 'latitude']].drop_duplicates(),
    geometry=gpd.points_from_xy(alertario_df['longitude'].drop_duplicates(), alertario_df['latitude'].drop_duplicates()),
    crs=CRS_GEOGRAPHIC
).to_crs(CRS_PROJECTED)

# Create a mapping from original lat/lon to projected x/y for faster lookup
proj_coords_map = {}
for idx, row in temp_stations_gdf.iterrows():
    # Find original lat/lon from stations_df that corresponds to this geometry
    # This part is a bit tricky if lat/lon are not unique identifiers. Assume they are for now.
    # A robust way is to merge projected coords back based on id_estacao.
    # For now, let's assume we can map back if needed, or better, add proj coords to alertario_df
    pass # This mapping part needs careful implementation.
'''

# Simpler: add projected coords directly to alertario_df for stations present
# Convert alertario_df station points to projected CRS for interpolation
alertario_gdf_points = gpd.GeoDataFrame(
    alertario_df,
    geometry=gpd.points_from_xy(alertario_df.longitude, alertario_df.latitude),
    crs=CRS_GEOGRAPHIC
).to_crs(CRS_PROJECTED)
alertario_df['longitude_proj'] = alertario_gdf_points.geometry.x
alertario_df['latitude_proj'] = alertario_gdf_points.geometry.y


# Store rasterized rainfall data: dictionary {timestamp: 2D_grid_array}
# all_timestamps = sorted(alertario_df['timestamp'].unique())
all_timestamps = sorted(alertario_df.sort_values('timestamp').iloc[-50000:]['timestamp'].unique())
rainfall_grids_dict = {} # Will store {timestamp: 2D numpy array}

print("Interpolating rainfall data onto grid...")
for i, ts in enumerate(all_timestamps):
    if (i+1) % 100 == 0:
        print(f"  Processed {i+1}/{len(all_timestamps)} timestamps...", end='\r')
    
    current_ts_data = alertario_df[alertario_df['timestamp'] == ts]
    
    # Get rainfall values for valid grid cells
    interpolated_rainfall_flat_valid = interpolate_rainfall_to_grid_idw(
        current_ts_data,
        grid_x_flat_valid,
        grid_y_flat_valid,
        city_mask_bool.ravel()[valid_grid_indices_flat] # Pass the mask for valid cells only
    )
    
    # Create the full grid and fill in the valid cell values
    full_grid_rainfall = np.zeros(GRID_HEIGHT_CELLS * GRID_WIDTH_CELLS, dtype=float)
    full_grid_rainfall[valid_grid_indices_flat] = interpolated_rainfall_flat_valid
    
    rainfall_grid_2d = full_grid_rainfall.reshape((GRID_HEIGHT_CELLS, GRID_WIDTH_CELLS))
    rainfall_grids_dict[ts] = rainfall_grid_2d

print(f"Finished rainfall interpolation. {len(rainfall_grids_dict)} rainfall grids created.")

# Example: Visualize one rainfall grid
if rainfall_grids_dict:
    example_ts = list(rainfall_grids_dict.keys())[len(all_timestamps)//2] # pick a timestamp from middle
    plt.figure(figsize=(10, 8))
    plt.imshow(rainfall_grids_dict[example_ts], cmap='Blues', origin='upper',
               extent=(minx, maxx, miny, maxy)) # Use extent for geographic context
    city_boundary_gdf.plot(ax=plt.gca(), facecolor='none', edgecolor='red', linewidth=0.5)
    plt.colorbar(label='Interpolated Rainfall (mm/15min)')
    plt.title(f'Interpolated Rainfall at {example_ts}')
    plt.xlabel("Easting (m)")
    plt.ylabel("Northing (m)")
    plt.show()
else:
    print("No rainfall grids were generated.")

---

### **5. Target Variable Generation (Python Cell)**
### 5.1 Rasterize Flood Occurrences
For each 15-minute interval corresponding to a rainfall grid, create a target grid.
A grid cell is marked as "flooded" (1) if it's within `FLOOD_SPATIAL_RADIUS_METERS` of a reported flood event
and the timestamp falls within the event's `[data_inicio - buffer, data_fim + buffer]`.

In [ ]:
# Align flood occurrences to 15-min intervals of our rainfall grids
# This means for a flood event, we identify all 15-min timestamps it spans.
target_flood_grids_dict = {ts: np.zeros((GRID_HEIGHT_CELLS, GRID_WIDTH_CELLS), dtype='uint8') for ts in rainfall_grids_dict.keys()}

print("Rasterizing flood occurrences onto grid...")
for i, (idx, event) in enumerate(ocorrencias_gdf.sort_values('data_inicio').iloc[-200:].iterrows()):
    event_start = event['data_inicio'] - timedelta(minutes=FLOOD_TEMPORAL_BUFFER_MINUTES_BEFORE)
    event_end = event['data_fim'] + timedelta(minutes=FLOOD_TEMPORAL_BUFFER_MINUTES_AFTER)
    
    # Create a circular buffer around the flood point
    # Note: If event.geometry is None or empty, skip
    if event.geometry is None or event.geometry.is_empty:
        continue
    flood_influence_geom = event.geometry.buffer(FLOOD_SPATIAL_RADIUS_METERS)
    
    # Rasterize this influence area
    # Note: This rasterizes one event. We need to OR these together for each timestamp.
    event_raster_influence = rasterize(
        [(flood_influence_geom, 1)],
        out_shape=(GRID_HEIGHT_CELLS, GRID_WIDTH_CELLS),
        transform=transform,
        fill=0,
        dtype='uint8'
    )
    
    # Find all 15-min timestamps affected by this event
    # Iterate through timestamps in target_flood_grids_dict
    for ts in target_flood_grids_dict.keys():
        # Check if the timestamp 'ts' (which is the START of a 15-min interval)
        # falls within the buffered event duration.
        # Interval: [ts, ts + 15_min_delta)
        ts_interval_end = pd.Timestamp(ts) + timedelta(minutes=TIME_RESOLUTION_MINUTES)
        
        # Check for overlap: (StartA <= EndB) and (EndA >= StartB)
        if event_start.tz_localize(None) < ts_interval_end and event_end.tz_localize(None) > pd.Timestamp(ts):
            target_flood_grids_dict[ts] = np.maximum(target_flood_grids_dict[ts], event_raster_influence)

    if i % 10 == 0 or i + 1 == len(ocorrencias_gdf.iloc[:200]):
        print(f'Events processed: {i + 1}/{len(ocorrencias_gdf.iloc[:200])}', end='\r')
        
# Apply city mask to target grids (ensure floods are only within the city)
for i, ts in enumerate(target_flood_grids_dict.keys()):
    target_flood_grids_dict[ts] = target_flood_grids_dict[ts] * city_mask_bool

    if i % 10 == 0 or i + 1 == len(target_flood_grids_dict):
        print(f'Events processed: {i + 1}/{len(target_flood_grids_dict)}', end='\r')

print(f"Finished flood occurrences rasterization.")
num_flood_pixels_total = sum(np.sum(grid) for grid in target_flood_grids_dict.values())
print(f"Total number of 'flooded' grid cell-timesteps: {num_flood_pixels_total}")

# Example: Visualize one target grid
if target_flood_grids_dict and rainfall_grids_dict: # Check if rainfall_grids_dict is not empty
    # Find a timestamp that has some flood pixels, if possible
    example_ts_flood = None
    for ts, grid in target_flood_grids_dict.items():
        if np.sum(grid) > 0:
            example_ts_flood = ts
            break
    if example_ts_flood is None: # if no floods, pick the same as rainfall example
        example_ts_flood = list(rainfall_grids_dict.keys())[len(all_timestamps)//2] if all_timestamps else None

    if example_ts_flood:
        plt.figure(figsize=(10, 8))
        plt.imshow(target_flood_grids_dict[example_ts_flood], cmap='Reds', origin='upper',
                   extent=(minx, maxx, miny, maxy))
        city_boundary_gdf.plot(ax=plt.gca(), facecolor='none', edgecolor='black', linewidth=0.5)
        plt.colorbar(label='Flood Label (1=Flood)')
        plt.title(f'Target Flood Grid at {example_ts_flood}')
        plt.xlabel("Easting (m)")
        plt.ylabel("Northing (m)")
        plt.show()
    else:
        print("Could not find an example timestamp for flood grid visualization.")

else:
    print("Cannot visualize target grid as rainfall_grids_dict is empty or not populated.")

---

### **6. Prepare Data for CNN (Python Cell)**
### 6.1 Create Input Sequences and Target Maps
Iterate through timestamps. For each timestamp `t`, create an input sequence of rainfall grids from `t - INPUT_SEQUENCE_LENGTH*15min` to `t-15min`. The target is the flood grid at time `t`.
Handle feature scaling.

In [7]:
X_sequences = []
y_targets = []

# Ensure timestamps are sorted for sequence creation
sorted_timestamps = sorted(rainfall_grids_dict.keys())

for i in range(INPUT_SEQUENCE_LENGTH -1, len(sorted_timestamps)):
    current_target_ts = sorted_timestamps[i]
    
    # Check if this timestamp should be used for prediction
    # (e.g. if we only want to predict for timestamps where flood events *could* occur)
    # For now, we prepare data for all available timestamps
    
    input_seq_ts = sorted_timestamps[i - (INPUT_SEQUENCE_LENGTH -1) : i+1] # t-(N-1) to t
    
    # The target is at sorted_timestamps[i] (current_target_ts)
    # The input sequence is from sorted_timestamps[i - INPUT_SEQUENCE_LENGTH + 1] to sorted_timestamps[i]
    # For predicting risk at time T, we use rainfall up to time T.
    # So if target is at T, input uses T, T-1, T-2, T-3 (for sequence length 4)

    # Let's adjust for clarity: predict risk at 'current_target_ts' using rainfall from
    # 'current_target_ts - (N-1)*delta_t' up to 'current_target_ts'.
    # If INPUT_SEQUENCE_LENGTH = 1, use rainfall at 'current_target_ts'.
    # If INPUT_SEQUENCE_LENGTH = 4, use rainfall at [ts-3, ts-2, ts-1, ts].
    
    sequence_grids = []
    valid_sequence = True
    for lag in range(INPUT_SEQUENCE_LENGTH - 1, -1, -1): # From ts-(N-1)*dt up to ts
        ts_for_feature = current_target_ts - timedelta(minutes=lag * TIME_RESOLUTION_MINUTES)
        if ts_for_feature in rainfall_grids_dict:
            sequence_grids.append(rainfall_grids_dict[ts_for_feature])
        else:
            valid_sequence = False # Missing data in the lookback window
            break
    
    if valid_sequence and current_target_ts in target_flood_grids_dict:
        # sequence_grids are now [Rain(t-(N-1)), Rain(t-(N-2)), ..., Rain(t)]
        # We want them in chronological order for ConvLSTM: [Rain(t-(N-1)), ..., Rain(t)]
        # The current loop structure already does this if we append.
        # My loop was: input_seq_ts = sorted_timestamps[i - INPUT_SEQUENCE_LENGTH + 1 : i + 1]
        # This gives [ts_at_lag_N-1, ..., ts_at_lag_0]
        
        # Let's re-verify sequence generation:
        # To predict for time T_i, we need rainfall maps from T_{i-N+1} ... T_i
        # T_i is `current_target_ts`
        
        idx_current_target_ts = sorted_timestamps.index(current_target_ts)
        if idx_current_target_ts < INPUT_SEQUENCE_LENGTH -1:
            continue # Not enough past data for a full sequence

        input_rain_sequence = []
        for k in range(INPUT_SEQUENCE_LENGTH):
            # Timestamp for k-th map in sequence (0 is earliest, N-1 is latest)
            # T_target - (N-1-k)*dt
            ts_feature_map = current_target_ts - timedelta(minutes=(INPUT_SEQUENCE_LENGTH - 1 - k) * TIME_RESOLUTION_MINUTES)
            if ts_feature_map in rainfall_grids_dict:
                input_rain_sequence.append(rainfall_grids_dict[ts_feature_map])
            else:
                input_rain_sequence = [] # Invalidate sequence
                break
        
        if input_rain_sequence: # If sequence is complete
            X_sequences.append(np.stack(input_rain_sequence, axis=0)) # Stack along time axis
            y_targets.append(target_flood_grids_dict[current_target_ts])

if not X_sequences:
    raise ValueError("No sequences were generated. Check data alignment, timestamps, and INPUT_SEQUENCE_LENGTH.")

X_all = np.array(X_sequences) # Shape: (num_samples, seq_len, height, width)
y_all = np.array(y_targets)   # Shape: (num_samples, height, width)

# Add channel dimension for CNN: (num_samples, seq_len, height, width, 1_channel)
X_all = np.expand_dims(X_all, axis=-1)
# Target for pixel-wise binary_crossentropy: (num_samples, height, width, 1_channel)
y_all = np.expand_dims(y_all, axis=-1)

print(f"Generated X shape: {X_all.shape}") # (Samples, TimeSteps, H, W, Channels)
print(f"Generated y shape: {y_all.shape}") # (Samples, H, W, Channels)

# --- Feature Scaling (Rainfall) ---
# Scale only the rainfall data (X_all).
# Reshape for scaler: (num_samples * seq_len * height * width, 1) if scaling globally
# Or scale per image, or per sequence. Simpler to scale globally for now.
original_shape_X = X_all.shape
X_all_flat = X_all.reshape(-1, 1)
RAINFALL_SCALER.fit(X_all_flat) # Fit on all data (can lead to leakage if not careful with split)
# Better: fit on training data only. For now, this is simpler for one-pass plan.
X_all_scaled_flat = RAINFALL_SCALER.transform(X_all_flat)
X_all = X_all_scaled_flat.reshape(original_shape_X)

print("Rainfall data scaled.")

Generated X shape: (1566, 4, 75, 144, 1)
Generated y shape: (1566, 75, 144, 1)
Rainfall data scaled.


### 6.2 Chronological Data Split
Split data into training, validation, and test sets based on time.
This is crucial to prevent data leakage and evaluate true forecasting ability.

In [8]:
num_samples = X_all.shape[0]
train_end_idx = int(num_samples * (1 - VALIDATION_SPLIT_RATIO - TEST_SPLIT_RATIO))
val_end_idx = int(num_samples * (1 - TEST_SPLIT_RATIO))

X_train, y_train = X_all[:train_end_idx], y_all[:train_end_idx]
X_val, y_val = X_all[train_end_idx:val_end_idx], y_all[train_end_idx:val_end_idx]
X_test, y_test = X_all[val_end_idx:], y_all[val_end_idx:]

# If RAINFALL_SCALER was not fit on all data, fit it here on X_train only
# And then transform X_train, X_val, X_test.
# For this plan, we assumed fitting on all for simplicity, acknowledge this limitation.

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# Verify no empty splits
if X_train.shape[0] == 0 or X_val.shape[0] == 0 or X_test.shape[0] == 0:
    raise ValueError("One of the data splits is empty. Adjust split ratios or check data quantity.")

X_train shape: (939, 4, 75, 144, 1), y_train shape: (939, 75, 144, 1)
X_val shape: (313, 4, 75, 144, 1), y_val shape: (313, 75, 144, 1)
X_test shape: (314, 4, 75, 144, 1), y_test shape: (314, 75, 144, 1)


---

### **7. CNN Model Architecture (Python Cell)**
Define a ConvLSTM model. The input is a sequence of rainfall grids, and the output is a single flood risk grid for the next timestamp.

In [ ]:
def build_convlstm_model(input_shape, num_filters_convlstm=32, num_filters_conv=32, kernel_size_convlstm=(3,3), kernel_size_conv=(3,3)):
    """
    Builds a ConvLSTM model for spatio-temporal prediction.
    Input shape: (seq_len, height, width, channels)
    Output shape: (height, width, 1) (pixel-wise probability)
    """
    inputs = Input(shape=input_shape) # e.g., (INPUT_SEQUENCE_LENGTH, GRID_HEIGHT_CELLS, GRID_WIDTH_CELLS, 1)

    # First ConvLSTM layer
    # return_sequences=True if stacking ConvLSTMs, False if it's the last one before spatial processing
    x = ConvLSTM2D(
        filters=num_filters_convlstm,
        kernel_size=kernel_size_convlstm,
        padding='same',
        return_sequences=True, # Keep sequences if more ConvLSTMs follow
        activation='relu' # Or use BatchNormalization + Activation
    )(inputs)
    x = BatchNormalization()(x)
    # x = Dropout(0.2)(x) # Optional

    # Second ConvLSTM layer (optional, can help learn more complex temporal patterns)
    x = ConvLSTM2D(
        filters=num_filters_convlstm * 2, # Increase filters
        kernel_size=kernel_size_convlstm,
        padding='same',
        return_sequences=False, # Output is the last state's spatial map
        activation='relu'
    )(x)
    x = BatchNormalization()(x)
    # x = Dropout(0.2)(x)

    # The output of ConvLSTM (if return_sequences=False) is (batch, height, width, filters)
    # Now, apply 2D Convolutions to refine spatial features and reduce to 1 channel for output.
    x = Conv2D(
        filters=num_filters_conv,
        kernel_size=kernel_size_conv,
        padding='same',
        activation='relu'
    )(x)
    x = BatchNormalization()(x)
    # x = Dropout(0.2)(x)

    # Output layer: 1x1 Convolution to get a single channel map (flood probability)
    outputs = Conv2D(
        filters=1, # Single output channel
        kernel_size=(1, 1),
        activation='sigmoid', # For binary classification (flood/no-flood) per pixel
        padding='same'
    )(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Get input shape from training data
# X_train shape is (Samples, TimeSteps, H, W, Channels)
# Keras Input needs (TimeSteps, H, W, Channels)
cnn_input_shape = X_train.shape[1:] 

model = build_convlstm_model(cnn_input_shape)
model.summary()

---

### **8. Model Training (Python Cell)**
Compile and train the model. Use binary cross-entropy for the pixel-wise classification.
Consider class weights for imbalanced flood data.

In [ ]:
# --- Handling Class Imbalance (Optional but Recommended) ---
# Calculate class weights for the target variable (y_train)
# Flatten y_train to count 0s and 1s across all pixels and samples
y_train_flat_for_weights = y_train.reshape(-1)
num_neg = np.sum(y_train_flat_for_weights == 0)
num_pos = np.sum(y_train_flat_for_weights == 1)
total = num_neg + num_pos

if num_pos == 0:
    print("WARNING: No positive samples (floods) in the training target data. Class weights cannot be computed meaningfully.")
    class_weight_dict = None # No weighting if no positive class
else:
    # Weight for class 0 (no flood)
    weight_for_0 = (1 / num_neg) * (total / 2.0) if num_neg > 0 else 0
    # Weight for class 1 (flood)
    weight_for_1 = (1 / num_pos) * (total / 2.0) if num_pos > 0 else 0
    class_weight_dict = {0: weight_for_0, 1: weight_for_1}
    print(f"Class weights: {class_weight_dict}")


# --- Loss function ---
# For pixel-wise binary classification, BinaryCrossentropy is standard.
# If using class weights, the model's `fit` method handles it.
# Alternative: Focal Loss for highly imbalanced data. For now, use BCE with class_weight.
loss_function = tf.keras.losses.BinaryCrossentropy()

# (Optional) Use a Weighted Loss Function Instead
# def weighted_bce(y_true, y_pred):
#     # Flatten y_true and y_pred
#     y_true_f = tf.reshape(y_true, [-1])
#     y_pred_f = tf.reshape(y_pred, [-1])
    
#     # Define weights for each class
#     weight_for_0 = (1 / num_neg) * (total / 2.0) if num_neg > 0 else 0
#     weight_for_1 = (1 / num_pos) * (total / 2.0) if num_pos > 0 else 0

#     weights = y_true_f * weight_for_1 + (1 - y_true_f) * weight_for_0
#     bce = tf.keras.backend.binary_crossentropy(y_true_f, y_pred_f)
#     weighted_bce = weights * bce
#     return tf.reduce_mean(weighted_bce)


# --- Metrics ---
# AUC-PR is good for imbalanced data. Also track Precision and Recall for the positive class.
# Keras metrics are typically for the whole batch, might need custom for per-pixel positive class.
# For simplicity, start with standard metrics.
metrics = [
    tf.keras.metrics.Precision(name='precision', thresholds=0.5), # Default threshold 0.5
    tf.keras.metrics.Recall(name='recall', thresholds=0.5),
    tf.keras.metrics.AUC(name='auc_roc', curve='ROC'),
    tf.keras.metrics.AUC(name='auc_pr', curve='PR') # Area Under Precision-Recall Curve
]

# --- Compile Model ---
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss=loss_function,
    metrics=metrics
)

# --- Callbacks ---
early_stopping = EarlyStopping(
    monitor='val_auc_pr', # Monitor validation AUC-PR
    patience=10,         # Stop if no improvement after 10 epochs
    mode='max',          # AUC-PR should be maximized
    restore_best_weights=True
)
model_checkpoint = ModelCheckpoint(
    MODEL_OUTPUT_PATH,
    monitor='val_auc_pr',
    save_best_only=True,
    mode='max',
    verbose=1
)

# --- Train Model ---
print("Starting model training...")
EPOCHS = 10
class_weight_dict = None
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, model_checkpoint],
    class_weight=class_weight_dict if num_pos > 0 else None # Pass class weights if computed
)

print("Model training finished.")

# Plot training history
pd.DataFrame(history.history).plot(figsize=(12, 8))
plt.title("Model Training History")
plt.xlabel("Epoch")
plt.ylabel("Metric Value")
plt.grid(True)
plt.show()

Class weights: {0: 0.5004957480168846, 1: 504.7884519661523}
Starting model training...
Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 27s/step - auc_pr: 0.0014 - auc_roc: 0.5032 - loss: 0.7506 - precision: 0.0012 - recall: 0.3895 
Epoch 1: val_auc_pr improved from -inf to 0.00010, saving model to ./cnn_flood_model.keras
30/30 ━━━━━━━━━━━━━━━━━━━━ 881s 29s/step - auc_pr: 0.0014 - auc_roc: 0.5035 - loss: 0.7481 - precision: 0.0012 - recall: 0.3865 - val_auc_pr: 9.6142e-05 - val_auc_roc: 0.5000 - val_loss: 0.6036 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 28s/step - auc_pr: 0.0024 - auc_roc: 0.5824 - loss: 0.5552 - precision: 0.0046 - recall: 0.1363 
Epoch 2: val_auc_pr did not improve from 0.00010
30/30 ━━━━━━━━━━━━━━━━━━━━ 903s 30s/step - auc_pr: 0.0024 - auc_roc: 0.5818 - loss: 0.5544 - precision: 0.0047 - recall: 0.1357 - val_auc_pr: 8.7777e-05 - val_auc_roc: 0.4833 - val_loss: 0.5354 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Ep

---

### **9. Model Evaluation (Python Cell)**
Evaluate the trained model on the test set.
Calculate metrics like Precision, Recall, F1-score, CSI, specifically for the "flood" class.
Visualize some predictions against ground truth.

In [ ]:
# Load the best model saved by ModelCheckpoint
print(f"Loading best model from: {MODEL_OUTPUT_PATH}")
best_model = tf.keras.models.load_model(MODEL_OUTPUT_PATH) # Add custom_objects if custom layers/losses were used

# --- Evaluate on Test Set ---
print("Evaluating model on test set...")
test_loss, test_precision, test_recall, test_auc_roc, test_auc_pr = best_model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Set Performance:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Precision (default threshold 0.5): {test_precision:.4f}")
print(f"  Recall (default threshold 0.5): {test_recall:.4f}")
print(f"  AUC (ROC): {test_auc_roc:.4f}")
print(f"  AUC (PR): {test_auc_pr:.4f}")

# --- More Detailed Metrics (CSI, F1 for flood class) ---
y_pred_proba_test = best_model.predict(X_test) # Probabilities
y_pred_test_binary = (y_pred_proba_test > 0.5).astype(int) # Apply threshold

# Flatten for pixel-wise metrics
y_test_flat = y_test.ravel()
y_pred_test_binary_flat = y_pred_test_binary.ravel()

# Calculate for the positive class (flood=1)
from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report (pixel-wise, on test set, threshold 0.5):")
# Note: target_names=['No Flood', 'Flood'] assumes 0 and 1 are present. Handle if not.
# Check if both classes are present in y_test_flat and y_pred_test_binary_flat
unique_true = np.unique(y_test_flat)
unique_pred = np.unique(y_pred_test_binary_flat)

if len(unique_true) > 1 and len(unique_pred) > 0 : # Need at least one prediction if true has both
    print(classification_report(y_test_flat, y_pred_test_binary_flat, target_names=['No Flood', 'Flood']))
    cm = confusion_matrix(y_test_flat, y_pred_test_binary_flat)
    print("Confusion Matrix (pixel-wise):")
    print(cm)
    # TP = cm[1,1], FN = cm[1,0], FP = cm[0,1], TN = cm[0,0]
    TP = cm[1,1] if cm.shape == (2,2) else 0
    FN = cm[1,0] if cm.shape == (2,2) else np.sum(y_test_flat == 1) if TP==0 and np.sum(y_pred_test_binary_flat==1)==0 else 0
    FP = cm[0,1] if cm.shape == (2,2) else np.sum(y_pred_test_binary_flat == 1) if TP==0 and np.sum(y_test_flat==1)==0 else 0

    CSI = TP / (TP + FN + FP) if (TP + FN + FP) > 0 else 0 # Critical Success Index (Threat Score)
    print(f"  Critical Success Index (CSI/Threat Score): {CSI:.4f}")
else:
    print("Could not generate full classification report or CSI (likely due to one class dominating or missing in predictions/true labels).")
    print(f"Unique true labels in test: {unique_true}")
    print(f"Unique predicted labels in test: {unique_pred}")


# --- Visualize Predictions ---
num_viz_samples = min(5, X_test.shape[0]) # Visualize a few samples
if num_viz_samples > 0:
    print(f"\nVisualizing {num_viz_samples} predictions from test set...")
    fig, axes = plt.subplots(num_viz_samples, 3, figsize=(15, num_viz_samples * 5))
    if num_viz_samples == 1: axes = np.array([axes]) # Ensure axes is 2D array

    for i in range(num_viz_samples):
        # Last rainfall map in the input sequence for context
        last_rain_map = X_test[i, -1, :, :, 0] # Last map in sequence, first channel
        # Inverse transform if scaled
        # Assuming RAINFALL_SCALER was fit on reshaped data:
        # For visualization, we'd need to inverse transform just this slice correctly.
        # This can be tricky. Simplest is to plot the scaled version or re-scale for plotting.
        # For simplicity, plotting scaled for now.
        
        ax = axes[i, 0]
        im = ax.imshow(last_rain_map, cmap='Blues', origin='upper', 
                       extent=(minx, maxx, miny, maxy))
        city_boundary_gdf.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=0.5)
        ax.set_title(f'Sample {i}: Last Input Rain (Scaled)')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[i, 1]
        im = ax.imshow(y_test[i, :, :, 0], cmap='Reds', vmin=0, vmax=1, origin='upper',
                       extent=(minx, maxx, miny, maxy))
        city_boundary_gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.5)
        ax.set_title(f'Sample {i}: True Flood Mask')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[i, 2]
        im = ax.imshow(y_pred_proba_test[i, :, :, 0], cmap='Oranges', vmin=0, vmax=1, origin='upper',
                       extent=(minx, maxx, miny, maxy))
        city_boundary_gdf.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.5)
        ax.set_title(f'Sample {i}: Predicted Flood Prob.')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()
else:
    print("Test set is empty, cannot visualize predictions.")

---

### **10. Interpretation, Refinements & Next Steps (Markdown Cell)**
## Interpretation and Discussion

*   Analyze the model's performance: Where does it succeed? Where does it fail?
*   Are the predicted flood probabilities spatially coherent? Do they align with known vulnerable areas or heavy rainfall?
*   The `FLOOD_SPATIAL_RADIUS_METERS` and temporal buffers are critical hyperparameters for label generation. Sensitivity analysis on these could be valuable.
*   The definition of "flood" from `ocorrencias` is inherently noisy. The model is learning from these imperfect labels.

## Potential Refinements & Future Work

1.  **Static Features:** Incorporate static geographical features as additional input channels to the CNN (e.g., elevation, slope, land use/imperviousness, distance to rivers). These would be repeated for each timestep in the input sequence or concatenated after the ConvLSTM layers.
2.  **Hyperparameter Tuning:** Systematically tune `INPUT_SEQUENCE_LENGTH`, ConvLSTM/Conv2D filter sizes, kernel sizes, learning rate, batch size.
3.  **Advanced Architectures:**
    *   Explore Attention mechanisms within the ConvLSTM.
    *   Try different CNN backbones or U-Net like structures for segmentation.
4.  **Improved Interpolation:** Use Kriging or other advanced methods instead of IDW for rainfall, if computationally feasible.
5.  **Handling Imbalance:** Experiment with Focal Loss or oversampling/undersampling techniques for the target grids if class weights are insufficient.
6.  **Uncertainty Quantification:** Use dropout at inference time (Monte Carlo Dropout) or ensemble methods to estimate prediction uncertainty.
7.  **Meteorological Data:** Integrate other meteorological variables (temperature, wind, etc.) if available at sufficient resolution, likely as separate channels.
8.  **Model Calibration:** Calibrate the output probabilities to be more reliable.
9.  **PU Learning:** If under-reporting is severe, explore Positive-Unlabeled learning strategies for generating the target variable. This would be a significant change to the labeling process.


---

This one-step plan provides a full workflow. Each Python cell block would correspond to a cell in a Jupyter Notebook. This is ambitious for a single shot, especially the data rasterization and sequence generation, but it lays out all necessary components. Debugging and iteration will undoubtedly be needed during actual implementation.